# 08e -- Manual Extraction Rankings

**Purpose**: Ingest manually-extracted ranking sheets from
`RookieRankings_2026_ManualExtraction.xlsx` into `fact_rookie_rankings`.
Each Excel sheet is one `source_site` / `source_name`.

| Sheet | source_name | source_site | phase |
|---|---|---|---|
| rotoballer | RotoBaller | RotoBaller | post_draft |
| mystery_iono | mystery_iono | mystery_iono | post_draft |
| dynastyleaguefootball | DynastyLeagueFootball | DynastyLeagueFootball | pre_combine |
| fantasycalc | FantasyCalc | FantasyCalc | post_draft |
| fantasypros_idp | FantasyPros_IDP | FantasyPros | post_draft |

**DLF phase note**: `Rank_Date` is 2025-12-31, which is pre-combine — using `pre_combine`.

**FP IDP note**: FantasyPros IDP rookie rankings cannot be scraped via `requests` —
the page at `dynasty-rookies-idp.php` serves a full veteran draft board in its raw HTML
`ecrData`; the actual defensive rookie table is rendered client-side only. Data is
manually extracted into the Excel sheet instead. Migrated from 08a.

**Prerequisites**: `08_fact_rookie_rankings_seed.ipynb`, `01_dim_rookie_prospect.ipynb`

**Outputs**: `data/fact_rookie_rankings.parquet` (appended),
             `data/review_fuzzy_matches.csv` (if new review items found)

## 1 -- Setup & Config

In [ ]:
import hashlib
import re
import shutil
import tempfile
from dataclasses import dataclass
from datetime import date, datetime
from pathlib import Path

import numpy as np
import pandas as pd
from thefuzz import fuzz, process


@dataclass
class LeagueConfig:
    """Central config — all league rules live here, nowhere else."""
    draft_year: int = 2026
    total_cap: int = 500_000_000
    num_teams: int = 28
    num_conferences: int = 2
    initial_contract_years: int = 3
    extension_contract_years: int = 3
    fa_minimum_salary: int = 2_000_000
    data_dir: str = "data"
    review_dir: str = "data/review"          # all review CSVs live here
    fuzzy_auto_threshold: int = 90
    fuzzy_review_threshold: int = 70


CFG    = LeagueConfig()
DATA   = Path(CFG.data_dir)
REVIEW = Path(CFG.review_dir)
TODAY  = date.today().isoformat()
DATA.mkdir(exist_ok=True)
REVIEW.mkdir(parents=True, exist_ok=True)
ALIAS  = DATA / "dim_player_alias.parquet"   # persistent name->player_key decisions (08y/08z)

# Excel is in the same data dir; copy to a temp file in case it is open in Excel.
# tempfile.gettempdir() is portable — no hardcoded user path.
XLSX_PATH = DATA / "RookieRankings_2026_ManualExtraction.xlsx"
XLSX_TMP  = Path(tempfile.gettempdir()) / "RookieRankings_manual_tmp.xlsx"

## 2 -- Shared Helpers

In [ ]:
def clean_player_name(name: str) -> str:
    # Normalize for matching: remove periods, collapse whitespace, lowercase.
    if pd.isna(name):
        return ""
    s = str(name).strip()
    s = s.replace(".", "").replace(" ", " ")
    s = s.replace("‘", "'").replace("’", "'").replace("`", "'")
    return " ".join(s.split()).lower()


def generate_player_key(name: str, position: str, school: str) -> str:
    # Deterministic 12-char MD5 hash -- matches keys generated in 01.
    raw = f"{clean_player_name(name)}|{str(position).upper().strip()}|{str(school).strip().lower()}"
    return hashlib.md5(raw.encode()).hexdigest()[:12]


assert clean_player_name("J.K. Dobbins") == "jk dobbins"
assert clean_player_name("D'Andre Swift") == "d'andre swift"
print("Helpers OK")

## 3 -- add_players_from_source

In [ ]:
def add_players_from_source(
    new_players_df: pd.DataFrame,
    source_name: str,
    draft_year: int = CFG.draft_year,
    auto_threshold: int = CFG.fuzzy_auto_threshold,
    review_threshold: int = CFG.fuzzy_review_threshold,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Fuzzy-match new player names against dim_rookie_prospect.
    # Returns (updated_dim_rookie_prospect, review_df).
    # new_players_df must have: player_name; optionally position_raw, school_raw.
    # NOTE: review_df is returned to caller -- file write is the caller's responsibility.
    dim_rp     = pd.read_parquet(DATA / "dim_rookie_prospect.parquet")
    pos_map    = pd.read_parquet(DATA / "dim_position.parquet")
    school_map = pd.read_parquet(DATA / "dim_school.parquet")

    existing_names  = dim_rp["player_name_clean"].tolist()
    existing_lookup = dict(zip(dim_rp["player_name_clean"], dim_rp["player_key"]))

    # Persistent decisions: (name_clean, position_raw) already resolved in a prior
    # review -> never ask again (dim_player_alias, built by 08y, appended by 08z).
    alias_keys = set()
    if ALIAS.exists():
        _al = pd.read_parquet(ALIAS)
        alias_keys = set(zip(_al["name_clean"], _al["position_raw"]))

    new_rows, review_rows, auto_alias_rows = [], [], []
    auto_matched = already_exists = alias_resolved = 0

    for _, row in new_players_df.iterrows():
        name_clean = clean_player_name(row["player_name"])
        pos_key    = str(row.get("position_raw", "")).upper().strip()

        if name_clean in existing_lookup:
            already_exists += 1
            continue

        # Already decided in a past review -> skip silently (no repeat review).
        if (name_clean, pos_key) in alias_keys:
            alias_resolved += 1
            continue

        best_match, score = ("", 0)
        if existing_names:
            best_match, score = process.extractOne(
                name_clean, existing_names, scorer=fuzz.token_sort_ratio
            )

        if score >= auto_threshold:
            # High-confidence auto-link. Record the alias so (a) ingest can attribute
            # this variant's ranking to the matched player_key instead of dropping it,
            # and (b) future runs skip it. Same fix as a manual "match" decision.
            auto_matched += 1
            mkey = existing_lookup.get(best_match, "")
            if mkey:
                auto_alias_rows.append({
                    "name_clean": name_clean, "position_raw": pos_key,
                    "player_key": mkey, "decision": "auto",
                    "source_example": source_name, "decided_date": TODAY,
                })
        elif score >= review_threshold:
            review_rows.append({
                "new_name":        row["player_name"],
                "new_name_clean":  name_clean,
                "new_position":    row.get("position_raw", ""),
                "new_school":      row.get("school_raw", ""),
                "best_match_name": best_match,
                "best_match_key":  existing_lookup.get(best_match, ""),
                "fuzzy_score":     score,
                "action":          "",  # fill: "match" or "new"
                "source":          source_name,
            })
        else:
            # New prospect -- add to dim_rookie_prospect
            pos_raw    = str(row.get("position_raw", "")).upper().strip()
            school_raw = str(row.get("school_raw", "")).strip()
            pos_match  = pos_map[pos_map["position_raw"] == pos_raw]
            sch_match  = school_map[school_map["school_raw"] == school_raw]
            pkey       = generate_player_key(row["player_name"], pos_raw, school_raw)
            new_rows.append({
                "player_key":        pkey,
                "player_name":       row["player_name"],
                "player_name_clean": name_clean,
                "position_raw":      pos_raw,
                "position_detail":   pos_match["position_detail"].iloc[0] if len(pos_match) else None,
                "position_group":    pos_match["position_group"].iloc[0]  if len(pos_match) else None,
                "side_of_ball":      pos_match["side_of_ball"].iloc[0]    if len(pos_match) else None,
                "fantasy_relevant":  pos_match["fantasy_relevant"].iloc[0] if len(pos_match) else False,
                "school_raw":        school_raw,
                "school_canonical":  sch_match["school_canonical"].iloc[0] if len(sch_match) else school_raw,
                "conference":        sch_match["conference"].iloc[0]       if len(sch_match) else "Unknown",
                "height_inches":     None,
                "weight":            None,
                "pfr_id":            None,
                "cfb_id":            None,
                "gsis_id":           pd.NA,
                "draft_year":        draft_year,
                "source":            source_name,
                "added_date":        TODAY,
            })
            existing_names.append(name_clean)
            existing_lookup[name_clean] = pkey

    if new_rows:
        dim_rp = pd.concat([dim_rp, pd.DataFrame(new_rows)], ignore_index=True)
        dim_rp.drop_duplicates(subset=["player_key"], inplace=True)
        dim_rp.to_parquet(DATA / "dim_rookie_prospect.parquet", index=False)

    # Persist auto-matches to the alias (append + dedup on the (name_clean, pos) key).
    if auto_alias_rows:
        aa = pd.DataFrame(auto_alias_rows)
        if ALIAS.exists():
            aa = pd.concat([pd.read_parquet(ALIAS), aa], ignore_index=True)
        aa.drop_duplicates(subset=["name_clean", "position_raw"], keep="last", inplace=True)
        aa.to_parquet(ALIAS, index=False)

    review_df = pd.DataFrame(review_rows) if review_rows else pd.DataFrame()

    print(f"Source: {source_name}")
    print(f"  Already in dim_rookie_prospect : {already_exists}")
    print(f"  Resolved via alias (no re-ask)  : {alias_resolved}")
    print(f"  Auto-matched (>={auto_threshold}) + aliased  : {auto_matched}")
    print(f"  New prospects added             : {len(new_rows)}")
    print(f"  Needs manual review             : {len(review_rows)}")

    return dim_rp, review_df

## 4 -- ingest_ranking_source

In [ ]:
def ingest_ranking_source(
    rankings_df: pd.DataFrame,
    source_name: str,
    source_site: str,
    phase: str,
    draft_year: int = CFG.draft_year,
    capture_date: str = TODAY,
    rank_date: str | None = None,
) -> pd.DataFrame:
    # Append one row per player to fact_rookie_rankings for a given source + phase.
    # Call add_players_from_source() first to ensure all players are in dim_rookie_prospect.
    # Players pending review (not yet in dim_rookie_prospect) are logged as unmatched and skipped.
    #
    # rankings_df required columns: player_name, global_rank
    # rankings_df optional columns: positional_rank, grade

    valid_phases = {"pre_combine", "post_combine", "post_draft"}
    if phase not in valid_phases:
        raise ValueError(f"phase must be one of {valid_phases}")

    dim_rp = pd.read_parquet(DATA / "dim_rookie_prospect.parquet")
    name_to_key = dict(zip(dim_rp["player_name_clean"], dim_rp["player_key"]))
    key_to_pfr  = dict(zip(dim_rp["player_key"],        dim_rp["pfr_id"]))

    # Fold in alias decisions so a matched name-variant (X resolved to prospect Y)
    # attributes its ranking to Y's player_key instead of being dropped as unmatched.
    # setdefault: real dim_rookie_prospect names always win over an alias.
    if ALIAS.exists():
        _al = pd.read_parquet(ALIAS)
        for _nc, _pk in zip(_al["name_clean"], _al["player_key"]):
            name_to_key.setdefault(_nc, _pk)

    dim_nfl = pd.read_parquet(DATA / "dim_nfl_players.parquet")
    pfr_to_gsis = dict(zip(dim_nfl["pfr_id"].dropna(), dim_nfl["gsis_id"].dropna()))

    rows, unmatched = [], []
    for _, row in rankings_df.iterrows():
        name_clean = clean_player_name(row["player_name"])
        pkey = name_to_key.get(name_clean)

        if pkey is None:
            unmatched.append(row["player_name"])
            continue

        pfr_id  = key_to_pfr.get(pkey)
        gsis_id = pfr_to_gsis.get(pfr_id) if pfr_id else None

        rows.append({
            "player_key":      pkey,
            "gsis_id":         gsis_id,
            "source_name":     source_name,
            "source_site":     source_site,
            "phase":           phase,
            "draft_year":      draft_year,
            "global_rank":     row.get("global_rank"),
            "positional_rank": row.get("positional_rank"),
            "grade":           row.get("grade"),
            "capture_date":    capture_date,
            "rank_date":       rank_date,
        })

    if unmatched:
        print(f"  WARN: {len(unmatched)} players pending review -- re-run after apply_review_decisions():")
        for name in unmatched[:10]:
            print(f"    {name}")
        if len(unmatched) > 10:
            print(f"    ... and {len(unmatched) - 10} more")

    new_df = pd.DataFrame(rows)
    if new_df.empty:
        print("  No rows to append.")
        return new_df

    new_df["global_rank"]     = new_df["global_rank"].astype("Int64")
    new_df["positional_rank"] = new_df["positional_rank"].astype("Int64")
    new_df["draft_year"]      = new_df["draft_year"].astype("Int64")
    new_df["rank_date"]       = new_df["rank_date"].where(new_df["rank_date"].notna(), other=None)

    existing = pd.read_parquet(DATA / "fact_rookie_rankings.parquet")
    combined = pd.concat([existing, new_df], ignore_index=True)
    _DEDUP   = ["player_key", "source_name", "phase", "draft_year"]
    # rank_date: preserve the first non-null value ever captured — never overwrite with a later scrape.
    combined["rank_date"] = (
        combined.groupby(_DEDUP, sort=False)["rank_date"]
        .transform(lambda s: s.dropna().iloc[0] if s.notna().any() else None)
    )
    combined.drop_duplicates(subset=_DEDUP, keep="last", inplace=True)
    combined.to_parquet(DATA / "fact_rookie_rankings.parquet", index=False)

    print(f"  Ingested: {len(rows)} rows | source={source_name} | site={source_site} | phase={phase}")
    print(f"  fact_rookie_rankings total: {len(combined)}")
    return new_df

## 5 -- Sheet Parsers

In [ ]:
def _pos_rank_from_group(df: pd.DataFrame, pos_col: str = "position_raw", rank_col: str = "global_rank") -> pd.Series:
    """Rank players within each position group by their global_rank order."""
    return df.groupby(pos_col)[rank_col].rank(method="first").astype(int)


def parse_rotoballer(df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    """Tier/Rank list; positional_rank computed from order within Pos group."""
    out = pd.DataFrame()
    out["player_name"]     = df["Player Name"].str.strip()
    out["position_raw"]    = df["Pos"].str.strip().str.upper()
    out["global_rank"]     = df["Rank"].astype(int)
    out["grade"]           = df["Rank"].astype(float)
    out["positional_rank"] = _pos_rank_from_group(out)
    rank_date = pd.to_datetime(df["Date"].iloc[0]).strftime("%Y-%m-%d")
    return out, rank_date


def _invert_lastname_firstname(s: str) -> str:
    # "Love, Jeremiyah" → "Jeremiyah Love"; pass-through if no comma.
    if pd.isna(s):
        return ""
    parts = [p.strip() for p in str(s).split(",", 1)]
    return f"{parts[1]} {parts[0]}" if len(parts) == 2 else parts[0]


def parse_mystery_iono(df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    """Name field is 'Lastname, Firstname'; invert before matching."""
    out = pd.DataFrame()
    out["player_name"]     = df["lastname_firstname"].apply(_invert_lastname_firstname)
    out["position_raw"]    = df["position_raw"].str.strip().str.upper()
    out["global_rank"]     = df["rank"].astype(int)
    out["grade"]           = df["rank"].astype(float)
    out["positional_rank"] = _pos_rank_from_group(out)
    rank_date = pd.to_datetime(df["date_added"].iloc[0]).strftime("%Y-%m-%d")
    return out, rank_date


def parse_dynastyleaguefootball(df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    """Pos column encodes both position and positional_rank: 'RB1', 'WR2', etc."""
    out = pd.DataFrame()
    out["player_name"]  = df["Name"].str.strip()
    pos_series          = df["Pos"].str.strip()
    # Extract alphabetic prefix → canonical position; extract trailing digit → pos rank.
    out["position_raw"]    = pos_series.str.extract(r"^([A-Za-z]+)")[0].str.upper()
    out["positional_rank"] = pos_series.str.extract(r"(\d+)$")[0].fillna(1).astype(int)
    out["global_rank"]     = df["Rank"].astype(int)
    out["grade"]           = df["Avg"].astype(float)   # consensus avg rank across 6 experts
    rank_date = pd.to_datetime(df["Rank_Date"].iloc[0]).strftime("%Y-%m-%d")
    return out, rank_date


def parse_fantasycalc(df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    """overallRank / positionRank are direct; grade = overallRank (no trade value stored here)."""
    out = pd.DataFrame()
    out["player_name"]     = df["name"].str.strip()
    out["position_raw"]    = df["position"].str.strip().str.upper()
    out["global_rank"]     = df["overallRank"].astype(int)
    out["positional_rank"] = df["positionRank"].astype(int)
    out["grade"]           = df["overallRank"].astype(float)
    rank_date = pd.to_datetime(df["date_added"].iloc[0]).strftime("%Y-%m-%d")
    return out, rank_date


def parse_fantasypros_idp(df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    """Manually extracted from FantasyPros dynasty-rookies-idp.php (client-side only).
    Player Name has '\\xa0(TEAM)' suffix; POS encodes position + positional_rank ('LB1', 'EDGE2')."""
    out = pd.DataFrame()
    # Strip non-breaking space + team abbreviation: "Sonny Styles\xa0(WAS)" → "Sonny Styles"
    out["player_name"]  = df["Player Name"].str.replace(r"\s*\(.*?\)\s*$", "", regex=True).str.strip()
    pos_series          = df["POS"].str.strip()
    out["position_raw"]    = pos_series.str.extract(r"^([A-Za-z]+)")[0].str.upper()
    out["positional_rank"] = pos_series.str.extract(r"(\d+)$")[0].fillna(1).astype(int)
    out["global_rank"]     = df["RK"].astype(int)
    out["grade"]           = df["RK"].astype(float)
    # No date column in this sheet — use today as rank_date.
    rank_date = TODAY
    return out, rank_date

## 6 -- Sources Config

In [ ]:
SOURCES = {
    "RotoBaller": {
        "sheet":       "rotoballer",
        "source_site": "RotoBaller",
        "phase":       "post_draft",
        "parser":      parse_rotoballer,
    },
    "mystery_iono": {
        "sheet":       "mystery_iono",
        "source_site": "mystery_iono",
        "phase":       "post_draft",
        "parser":      parse_mystery_iono,
    },
    "DynastyLeagueFootball": {
        "sheet":       "dynastyleaguefootball",
        "source_site": "DynastyLeagueFootball",
        "phase":       "pre_combine",  # Rank_Date=2025-12-31 → pre-combine
        "parser":      parse_dynastyleaguefootball,
    },
    "FantasyCalc": {
        "sheet":       "fantasycalc",
        "source_site": "FantasyCalc",
        "phase":       "post_draft",
        "parser":      parse_fantasycalc,
    },
    "FantasyPros_IDP": {
        "sheet":       "fantasypros_idp",
        "source_site": "FantasyPros",
        "phase":       "post_draft",
        "parser":      parse_fantasypros_idp,
    },
}

## 7 -- Ingestion Loop

In [ ]:
shutil.copy2(XLSX_PATH, XLSX_TMP)
xl = pd.ExcelFile(XLSX_TMP)

_all_reviews: list[pd.DataFrame] = []
_failed: list[str] = []

for source_name, cfg in SOURCES.items():
    print("=" * 60)
    print(f"Processing: {source_name}  [{cfg['phase']}]")

    try:
        df_raw = xl.parse(cfg["sheet"])
        df, rank_date = cfg["parser"](df_raw)
        print(f"  Parsed {len(df)} rows  |  rank_date={rank_date}")
    except Exception as e:
        print(f"  ERROR parsing: {e}")
        _failed.append(source_name)
        continue

    try:
        _, review = add_players_from_source(df, source_name=source_name)
        if not review.empty:
            _all_reviews.append(review)
    except Exception as e:
        print(f"  ERROR add_players: {e}")
        _failed.append(source_name)
        continue

    try:
        ingest_ranking_source(
            df,
            source_name=source_name,
            source_site=cfg["source_site"],
            phase=cfg["phase"],
            rank_date=rank_date,
        )
    except Exception as e:
        print(f"  ERROR ingesting: {e}")
        _failed.append(source_name)

if _failed:
    print(f"\nFAILED sources: {_failed}")
else:
    print("\nAll sources ingested successfully.")

## 8 -- Write Review CSV

In [ ]:
if _all_reviews:
    new_review = pd.concat(_all_reviews, ignore_index=True)
    rp = REVIEW / "review_fuzzy_matches.csv"
    if rp.exists():
        existing_rv = pd.read_csv(rp)
        combined_rv = pd.concat([existing_rv, new_review], ignore_index=True)
        # keep="first" preserves already-filled action values from prior review sessions.
        combined_rv.drop_duplicates(subset=["new_name", "source"], keep="first", inplace=True)
        n_added = len(combined_rv) - len(existing_rv)
        combined_rv.to_csv(rp, index=False)
        print(f"Review file updated: +{max(n_added, 0)} new rows ({len(combined_rv)} total) -> {rp}")
    else:
        new_review.to_csv(rp, index=False)
        print(f"Review file created: {rp} ({len(new_review)} rows)")
else:
    print("No fuzzy review items — all players matched cleanly.")

## 9 -- Validation

In [ ]:
fr  = pd.read_parquet(DATA / "fact_rookie_rankings.parquet")
sub = fr[fr["source_name"].isin(SOURCES.keys())]
print(f"Rows for this run : {len(sub)}")
print(sub.groupby(["source_name", "phase"]).size().rename("rows").to_string())
print()
print(f"fact_rookie_rankings total : {len(fr)} rows")
dupes = fr.duplicated(subset=["player_key", "source_name", "phase", "draft_year"]).sum()
print(f"Composite key duplicates   : {dupes} (expected 0)")
print()
print("Source breakdown:")
print(fr.groupby(["source_site", "source_name", "phase"]).size().rename("rows").to_string())